# 00 · Synthetic data — a visual tour (dev only, temporary)

**What this is for.** A visual way into the synthetic retail book that every stage is developed
against:
1. the book: who the clients are;
2. what they hold, and who picks products up;
3. how amounts behave;
4. how they move over time;
5. what the feature mart looks like;
6. the traps the framework must catch;
7. the ceiling no model can beat;
8. a few clients' stories.

Each section says what to look for; the sections follow Level 2 of the spec,
`docs/components/test_harness.md`. The review guide is `synthetic/REVIEW.md`. The guided EDA
notebook replaces this one once EDA exists.

**The checks.** The first cell prints whether every check row passes. The check tables and the
tables behind each chart are in the **appendix** at the end. A failing row is a generator bug, never
a tolerance to widen.

**How to use it.**
1. Generate the canonical dataset and its check tables once (about 6 minutes, plus 10–15 for the
   checks): `python -m synthetic --preset canonical --out data/01_raw/synthetic_canonical --checks`.
2. Restart the kernel and run all cells (a few minutes).

Amounts are in AED a month. The oracle columns are hidden truth, never features. Charts use a fixed
20% client sample (client ids divisible by 5), except the turnover and tail charts, which read every
client.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pyarrow.parquet as pq
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform

import synthetic
from synthetic.checks.suite import read_checks
from synthetic.mechanisms.traps import LEAK, trap_summary
from synthetic.output import load_truth, read_output
from synthetic.truth.oracle import auc, realised_target, window_probability

# The dataset lives next to the repo, never relative to the working directory.
DATA = Path(synthetic.__file__).resolve().parents[3] / "data" / "01_raw" / "synthetic_canonical"
truth = load_truth(str(DATA))  # fails loud if the dataset is older than the installed generator
spec = truth["spec"]
ORDER = list(spec["subsegments"])  # wealthiest first
PROFILES = ["private", "priority", "affluent", "mass"]
PROFILE_OF = {s: v["profile"] for s, v in spec["subsegments"].items()}
MAY = pd.Timestamp("2025-05-01").date()  # a month without seasonal effects, bonuses or school fees
pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_rows", 200)

# The 20% client sample, every column, every month: the mart and the hidden truth beside it.
mart = read_output(str(DATA), "mart", client_sample=5)
mart["profile"] = mart["subsegment"].map(PROFILE_OF)
oracle = read_output(str(DATA), "oracle", client_sample=5)
months = sorted(mart["snapshot_month"].unique())
first = mart[mart["snapshot_month"] == months[0]]
may = mart[mart["snapshot_month"] == MAY]

tables = read_checks(str(DATA))
n_rows = sum(len(t) for t in tables.values())
n_passed = sum(int(t["passed"].sum()) for t in tables.values())
print(
    f"generator {truth['generator_version']} · seed {truth['seed']}"
    f" · {truth['n_clients']:,} clients · {truth['n_months']} months from {truth['first_month']}"
)
print(f"checks: {n_passed:,} of {n_rows:,} rows pass, in {len(tables)} families (appendix A1)")

generator 0.5.0 · seed 0 · 100,000 clients · 24 months from 2024-10
checks: 1,309 of 1,309 rows pass, in 8 families (appendix A1)


## 1. The book: who the clients are
Nine subsegments in four profiles, ordered by wealth: Private 1%, Priority 4%, Affluent 9%,
Mass 86%. Asian expatriates dominate Mass, UAE nationals lead Private. Ages and incomes rise with
wealth; Private incomes are far apart. About 8–12% of clients leave each year, a little more often
lower down, and entries run 0.2 points above exits everywhere, so the mix stays flat.

In [2]:
mix = first["subsegment"].value_counts(normalize=True).reindex(ORDER).rename("share of clients")
px.bar(
    mix.reset_index(),
    x="subsegment",
    y="share of clients",
    color=mix.index.map(PROFILE_OF),
    log_y=True,
    labels={"color": "profile", "share of clients": "share of clients (log scale)"},
    title="Clients per subsegment, first panel month",
).show()
nationality = pd.crosstab(
    first["subsegment"], first["nationality_group"], normalize="index"
).reindex(ORDER)
px.bar(
    nationality,
    labels={"value": "share of the subsegment's clients", "nationality_group": "nationality"},
    title="Nationality mix per subsegment, first panel month",
).show()
# Boxes from percentiles (whiskers: 5th and 95th), so the notebook doesn't embed every client.
for column, axis, log in [
    ("age", "age (years)", False),
    ("income", "declared monthly income (AED, log scale)", True),
]:
    q = (
        first.groupby("subsegment")[column]
        .quantile([0.05, 0.25, 0.5, 0.75, 0.95])
        .unstack()
        .reindex(ORDER)
    )
    fig = go.Figure()
    for profile in PROFILES:
        part = q[q.index.map(PROFILE_OF) == profile]
        fig.add_trace(
            go.Box(
                x=part.index,
                lowerfence=part[0.05],
                q1=part[0.25],
                median=part[0.5],
                q3=part[0.75],
                upperfence=part[0.95],
                name=profile,
            )
        )
    fig.update_layout(
        title=f"{axis.split(' (')[0].capitalize()} per subsegment, first panel month"
        " (percentiles 5, 25, 50, 75, 95)",
        yaxis_title=axis,
        yaxis_type="log" if log else "linear",
        boxmode="overlay",
    )
    fig.update_xaxes(categoryorder="array", categoryarray=ORDER).show()

In [3]:
# Turnover read from the mart, every client (a 20% sample is too noisy for the small profiles): a
# leaver's last row is its last month on the books, a new client's first row its first month after
# joining. Rates are shares of the profile's clients that month.
keys = read_output(str(DATA), "mart", ["client_id", "snapshot_month", "subsegment"])
keys = keys.assign(profile=keys["subsegment"].map(PROFILE_OF))
book = keys.groupby(["snapshot_month", "profile"]).size()
last_rows = keys.drop_duplicates("client_id", keep="last")
first_rows = keys.drop_duplicates("client_id")
# A month without leavers (or joiners) counts as 0, not as missing.
exits = (
    last_rows[last_rows["snapshot_month"] < months[-1]]
    .groupby(["snapshot_month", "profile"])
    .size()
)
exits = exits.reindex(book.index, fill_value=0)
entries = (
    first_rows[first_rows["snapshot_month"] > months[0]]
    .groupby(["snapshot_month", "profile"])
    .size()
)
entries = entries.reindex(book.index, fill_value=0)
rates = pd.DataFrame({"exit rate": exits / book, "entry rate": entries / book}).reset_index()
rates = rates[(rates["snapshot_month"] > months[0]) & (rates["snapshot_month"] < months[-1])]
px.line(
    rates.melt(["snapshot_month", "profile"], ["exit rate", "entry rate"]),
    x="snapshot_month",
    y="value",
    color="profile",
    line_dash="variable",
    labels={"value": "share of the profile's clients"},
    title="Monthly exit and entry rates per profile",
).show()

## 2. Who holds what, and who picks it up
Holdings sit near the spec's shares and stay flat month after month; digital activity trends up by
design (the drift trap). Wealthier profiles hold more cards, term deposits and traditional
wealth; Priority leads digital wealth and home loans; personal loans peak in Affluent and Mass. By
declared income, the share holding a card, a personal or an auto loan jumps at 5,000 AED and a home
loan at 15,000 AED: below a product's minimum income the odds of taking it are a fifth. Clients
without a declared income count as above if they are Private, Priority, Retired or Other.

In [4]:
flags = [
    "casa_active_flag",
    "cc_flag",
    "cc_active_flag",
    "td_flag",
    "wealth_digital_flag",
    "wealth_trad_flag",
    "ebanking_flag",
    "digital_active_flag",
    "fx_active_flag",
]
monthly = mart.groupby(["snapshot_month", "profile"])[flags].mean().reset_index()
px.line(
    monthly.melt(["snapshot_month", "profile"], var_name="flag", value_name="share of clients"),
    x="snapshot_month",
    y="share of clients",
    color="flag",
    facet_col="profile",
    category_orders={"profile": PROFILES},
    title="Share of clients with each flag at 1, per month and profile",
).show()

In [5]:
loans = {
    "loan_personal": "loan_personal_cnt",
    "loan_auto": "loan_auto_cnt",
    "loan_home": "loan_home_cnt",
}
declared = first[first["income"].notna()]
held = pd.DataFrame(
    {"cc": declared["cc_flag"] == 1, **{k: declared[c] > 0 for k, c in loans.items()}}
)
band = pd.cut(declared["income"], np.geomspace(1_000, 200_000, 31))
by_income = held.groupby(band, observed=True).mean()
by_income.index = [interval.mid for interval in by_income.index]
fig = px.line(
    by_income,
    log_x=True,
    markers=True,
    labels={"index": "declared monthly income (AED, log scale)", "value": "share holding"},
    title="Share holding each product by declared income, first month; dotted: minimum incomes",
)
for minimum in sorted({p["min_income"] for p in spec["products"].values() if "min_income" in p}):
    fig.add_vline(x=minimum, line_dash="dot", annotation_text=f"{minimum:,} AED")
fig.show()

## 3. Amounts
Every amount is a median multiple of a hidden base (income for flows, wealth for balances), so
Private sits far above everyone. The tails are power laws: on log–log axes, far out, the share of
clients above a balance falls on a straight line, parallel to the spec's slope (−2 for the
current/savings balance). Neighbouring
subsegments overlap: about a quarter of MM clients hold more than the UM median. Card limits come
in product tiers that rise with income; revolvers run their cards fuller than transactors.
Private spends more on travel and moves money by transfer; Mass spends more on retail, cash and
remittances.

In [6]:
# The share of clients above each current/savings balance on log–log axes, May 2025, every client.
# Balances are divided by the subsegment's median so the subsegments share one scale (those with at
# least 1,000 clients). Far out, a power law is a straight line whose slope is minus the tail index.
balances = read_output(str(DATA), "mart", ["snapshot_month", "subsegment", "casa_bal"])
balances = balances[(balances["snapshot_month"] == MAY) & (balances["casa_bal"] > 0)]
lines = []
for subsegment, part in balances.groupby("subsegment"):
    if len(part) >= 1_000:
        values = np.sort(part["casa_bal"].to_numpy() / part["casa_bal"].median())[
            ::-1
        ]  # largest first
        rank = np.unique(np.geomspace(1, len(values), 200).astype(int))
        lines.append(
            pd.DataFrame(
                {
                    "balance / subsegment median": values[rank - 1],
                    "share of clients above": rank / len(values),
                    "subsegment": subsegment,
                }
            )
        )
fig = px.line(
    pd.concat(lines),
    x="balance / subsegment median",
    y="share of clients above",
    color="subsegment",
    log_x=True,
    log_y=True,
    category_orders={"subsegment": ORDER},
    title="Heavy tails: share of clients above each current/savings balance (log–log), May 2025",
)
tail = spec["levels"]["measures"]["casa_bal"]["tail"]
x = np.array([3.0, 300.0])
fig.add_trace(
    go.Scatter(
        x=x,
        y=0.1 * (x / 3) ** -tail,
        mode="lines",
        line_dash="dot",
        line_color="grey",
        name=f"slope −{tail}: the spec's tail index",
    )
)
fig.show()

In [7]:
mm_um = may[may["subsegment"].isin(["UM", "MM"]) & (may["casa_bal"] > 0)]
bins = np.linspace(1, 7, 61)
hist = pd.DataFrame(
    {
        s: np.histogram(np.log10(part["casa_bal"]), bins=bins)[0] / len(part)
        for s, part in mm_um.groupby("subsegment")
    },
    index=bins[:-1].round(1),
)
px.line(
    hist,
    labels={"index": "log10 current/savings balance (AED)", "value": "share of clients"},
    title="Neighbouring subsegments overlap: current/savings balance, UM vs MM, May 2025",
).show()

In [8]:
# Card limits against income: each card's limit is a product tier, set from the profile's multiple
# of true income (declared income shown; clients without one are left out).
holders = may[(may["cc_limit"] > 0) & may["income"].notna()]
px.scatter(
    holders.sample(min(len(holders), 4_000), random_state=0),
    x="income",
    y="cc_limit",
    color="profile",
    log_x=True,
    log_y=True,
    opacity=0.5,
    category_orders={"profile": PROFILES},
    labels={"income": "declared monthly income (AED)", "cc_limit": "card limit (AED)"},
    title="Card limit vs declared income, May 2025 (one dot a card; the bands are the tiers)",
).show()
cards = may.merge(
    oracle[["client_id", "snapshot_month", "revolver_flag"]], on=["client_id", "snapshot_month"]
)
cards = cards[cards["cc_limit"] > 0].assign(
    kind=lambda d: np.where(d["revolver_flag"] == 1, "revolver (hidden)", "transactor")
)
bins = np.linspace(0, 1, 21)
util = pd.DataFrame(
    {
        kind: np.histogram(part["cc_utilisation_ratio"], bins=bins)[0] / len(part)
        for kind, part in cards.groupby("kind")
    },
    index=bins[:-1].round(2),
)
px.bar(
    util,
    barmode="group",
    labels={"index": "utilisation (balance / limit)", "value": "share of cards"},
    title="Card utilisation, revolvers vs transactors, May 2025",
).show()

In [9]:
parts = {
    "outflow": [
        "outflow_casa_remittance_vol",
        "outflow_casa_cash_vol",
        "outflow_casa_bills_vol",
        "outflow_casa_transfers_vol",
    ],
    "card spend": [
        "spend_cc_travel_vol",
        "spend_cc_retail_vol",
        "spend_cc_online_vol",
        "spend_cc_dining_vol",
    ],
}
shares = []
for total, columns in parts.items():
    sums = mart.groupby("profile")[columns].sum().reindex(PROFILES)
    share = sums.div(sums.sum(axis=1), axis=0).rename(columns=lambda c: c.split("_")[-2])
    shares.append(share.reset_index().melt("profile", var_name="subcategory").assign(total=total))
px.bar(
    pd.concat(shares),
    x="profile",
    y="value",
    color="subcategory",
    facet_col="total",
    labels={"value": "share of the profile's total"},
    title="Where the money goes: outflow and card-spend mix per profile (all panel months)",
).show()

## 4. Time
Summer: travel spend and FX up, balances down, least for Mass. School fees in September, January
and April. Mobile logins jump at the app relaunch (2026-01): the drift monitoring must catch.

In [10]:
seasonal = (
    mart.groupby(["snapshot_month", "profile"])
    .agg(
        **{
            "mean card travel spend": ("spend_cc_travel_vol", "mean"),
            "mean bills paid": ("outflow_casa_bills_vol", "mean"),
            "median current/savings balance": ("casa_bal", "median"),
        }
    )
    .reset_index()
)
fig = px.line(
    seasonal.melt(["snapshot_month", "profile"], var_name="measure", value_name="AED"),
    x="snapshot_month",
    y="AED",
    color="profile",
    facet_row="measure",
    log_y=True,
    height=750,
    category_orders={"profile": PROFILES},
    title="Seasonality per profile (AED per client-month, log scale)",
)
fig.update_yaxes(matches=None).for_each_annotation(
    lambda a: a.update(text=a.text.split("=")[-1])
).show()
logins = mart.groupby("snapshot_month")["login_digital_mobile_cnt"].apply(lambda s: s[s > 0].mean())
px.line(
    logins.reset_index(),
    x="snapshot_month",
    y="login_digital_mobile_cnt",
    labels={"login_digital_mobile_cnt": "mean mobile logins"},
    title="Mean mobile logins a month of clients with any: the relaunch at 2026-01",
).show()

## 5. The mart
157 columns from one name grammar. Nulls appear only where retail data has them, for example
missing income, utilisation without a card, and rollups whose window reaches before the client
joined. The mart is collinear by construction, as real marts are: a measure's rollups move together
and identities hold exactly. Feature Checks must flag these clusters.

In [11]:
kinds = pd.DataFrame(truth["columns"]).set_index("name")["kind"]
nulls = mart.isna().mean().drop("profile")
nulls = (
    nulls[nulls > 0]
    .sort_values()
    .rename("share null")
    .to_frame()
    .assign(kind=lambda d: kinds.reindex(d.index))
)
px.bar(
    nulls.reset_index(names="column"),
    x="share null",
    y="column",
    color="kind",
    orientation="h",
    height=900,
    title="Share of rows that are null, per column with any null (all months)",
).show()

In [12]:
# A client who joined during the panel: its rollups stay null until the window fits inside the
# relationship (3 months for the 0_2 average, 6 for 0_5).
rows_per_client = mart.groupby("client_id")["snapshot_month"].agg(["min", "size"])
joiner = rows_per_client[
    (rows_per_client["min"] > months[2]) & (rows_per_client["size"] >= 12)
].index[0]
path = mart[mart["client_id"] == joiner]
px.line(
    path.melt("snapshot_month", ["casa_bal", "casa_bal_0_2_months_avg", "casa_bal_0_5_months_avg"]),
    x="snapshot_month",
    y="value",
    color="variable",
    markers=True,
    labels={"value": "AED"},
    title=f"Client {joiner}, joined {path['snapshot_month'].min():%Y-%m}: balance and its rollups",
).show()

In [13]:
# Rank correlations of the current/savings balance and inflow columns, May 2025, ordered so that
# similar columns sit together: blocks near 1 are rollups of one measure.
family = [c for c in mart.columns if c.startswith(("casa_bal", "inflow_casa"))]
corr = may[family].corr(method="spearman").fillna(0)
order = leaves_list(linkage(squareform(1 - corr.abs().to_numpy(), checks=False), "average"))
px.imshow(
    corr.iloc[order, order],
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    height=750,
    title="Collinear clusters: rank correlation of balance and inflow columns, May 2025",
).show()

## 6. Traps
Traps sit on credit-card acquisition and personal-loan opening. A leakage screen ranks every
column on its own: real columns fall below the oracle's AUC (dashed), while each target's trap
columns built for the charted window stand above it (card within 3 months: the two leaks, the
upstream target and the offer code, which leaks through being filled; loan next month: the process
leak and the shifted column). The card's own process leak and shifted limit are built for next
month, so at 3 months they sit below the line, as a leak built for another window does. The null
controls sit at 0.5. The low-fill amount sits near it: about 1% of clients have a value, so
noise moves its bar (on all 100k clients its AUC is 0.47). Constants have nothing to rank and are
left out.
Simpson's reversal: logins go with card uptake within each onboarding channel, against it pooled.

In [14]:
# Every numeric mart column alone against the realised target, all panel months pooled (20%
# sample), and each categorical trap column with nulls by whether it is filled (the offer code
# leaks that way). Strength is max(AUC, 1 − AUC), since a column can predict either way round;
# columns with a value in fewer than 300 eligible rows are left out. A column's AUC uses the rows
# where it has a value, the oracle's every eligible row: rollups are null in a joiner's first
# months, when card uptake peaks. Shown: the 15 strongest real columns, the target's own trap
# columns and the target-free ones (null controls, near-constant).
numeric = [
    c
    for c in mart.columns
    if c not in ("client_id", "snapshot_month") and pd.api.types.is_numeric_dtype(mart[c])
]
filled_only = [
    c
    for c in mart.columns
    if "_trap_" in c and c not in numeric and mart[c].isna().any() and mart[c].notna().any()
]
screens, chances = {}, {}
for target, w, label in [
    ("cc_acquire", 3, "card acquisition within 3 months"),
    ("loan_personal_open", 1, "personal-loan opening next month"),
]:
    y = realised_target(mart, target, w)
    use = y.notna().to_numpy()
    labels = y[use].to_numpy()
    chances[target] = window_probability(mart, oracle, truth, target, w)
    ceiling = auc(chances[target].to_numpy()[use], labels)
    rows = []
    for column in numeric:
        values = mart.loc[use, column].to_numpy(dtype=float, na_value=np.nan)
        known = ~np.isnan(values)
        if known.sum() >= 300 and np.unique(values[known]).size > 1:
            rows.append({"column": column, "AUC": auc(values[known], labels[known])})
    for column in filled_only:
        is_filled = mart.loc[use, column].notna().to_numpy(dtype=float)
        rows.append({"column": f"{column} is filled", "AUC": auc(is_filled, labels)})
    screen = pd.DataFrame(rows).assign(
        strength=lambda d: np.maximum(d["AUC"], 1 - d["AUC"]),
        kind=lambda d: np.where(d["column"].str.contains("_trap_"), "trap column", "real column"),
    )
    screens[target] = screen.assign(oracle=ceiling)
    real = screen[screen["kind"] == "real column"].nlargest(15, "strength")
    own = [
        t["column"]
        for t in spec["traps"]
        if t["trap_column"] and t["column"] and t["target"] in (target, None)
    ]
    own += [f"{c} is filled" for c in own if c in filled_only]
    shown = pd.concat([real, screen[screen["column"].isin(own)]]).sort_values(
        "strength", ascending=False
    )
    fig = px.bar(
        shown,
        x="strength",
        y="column",
        color="kind",
        orientation="h",
        height=650,
        category_orders={"column": shown["column"].tolist()},  # strongest on top
        range_x=[0.45, 1.0],
        title=f"Leakage screen, {label}: each column alone (AUC strength)",
    )
    fig.add_vline(x=ceiling, line_dash="dash", annotation_text=f"oracle {ceiling:.3f}")
    fig.add_vline(x=0.5, line_dash="dot", line_color="grey")
    fig.show()

In [15]:
card_chance = chances["cc_acquire"]  # the oracle's chance of taking a card within 3 months
frame = mart[["cc_flag", "onboarding_channel", "login_digital_mobile_cnt_0_2_months_avg"]].assign(
    chance=card_chance
)
frame = frame[(frame["cc_flag"] == 0) & frame["chance"].notna()]
frame = frame.assign(
    logins=pd.qcut(
        frame["login_digital_mobile_cnt_0_2_months_avg"].rank(method="first"), 10, labels=False
    )
    + 1
)
pooled = frame.groupby("logins")["chance"].mean().rename("pooled")
within = frame.groupby(["onboarding_channel", "logins"])["chance"].mean().unstack(0)
px.line(
    pd.concat([pooled, within], axis=1).reset_index(),
    x="logins",
    y=["pooled", "branch", "digital"],
    labels={
        "logins": "decile of mobile logins (3-month average)",
        "value": "mean chance",
        "variable": "clients",
    },
    title="Simpson's reversal: chance of taking a card within 3 months, by login decile",
).show()

## 7. The ceiling: what a perfect model would reach
The oracle knows each client's true chance, so its ranking is the best any model can do. The gains
curve shows what that means for a campaign: the share of card acquirers caught when targeting the
top x% of non-holders, for the oracle, the best real column, a simple rule and random. The leak
lies above the oracle: too good to be true, because it peeks ahead. The oracle is calibrated too:
its probabilities match realised rates. For regression, the oracle gives each card holder's
next-3-month card spend as a band; what happens falls inside it about 80% of the time.

In [16]:
y = realised_target(mart, "cc_acquire", 3)
use = y.notna().to_numpy()
labels = y[use].to_numpy()
screen = screens["cc_acquire"]
best = screen[screen["kind"] == "real column"].nlargest(1, "strength").iloc[0]
direction = 1 if best["AUC"] >= 0.5 else -1
eligible = mart.loc[
    use, list(dict.fromkeys([best["column"], "salary_transfer_flag", "income", LEAK]))
]
order_word = "highest" if direction == 1 else "lowest"
scores = {
    "oracle (true chance)": card_chance.to_numpy()[use],
    f"best real column: {best['column']}, {order_word} first": direction
    * eligible[best["column"]].to_numpy(float, na_value=np.nan),
    "rule: salary transfer and income ≥ 5,000": (
        (eligible["salary_transfer_flag"] == 1) & (eligible["income"] >= 5_000)
    ).to_numpy(float),
    "leak: preapproval score (peeks ahead)": eligible[LEAK].to_numpy(float, na_value=np.nan),
    "random": np.zeros(len(labels)),
}
tie = np.random.default_rng(0).random(len(labels))  # random order within ties
targeted = np.linspace(0, 1, 201)
curves = []
for name, score in scores.items():
    order = np.lexsort((tie, -np.nan_to_num(score, nan=-np.inf)))  # highest first, nulls last
    caught = np.concatenate([[0.0], np.cumsum(labels[order])]) / labels.sum()
    curves.append(
        pd.DataFrame(
            {
                "share of non-holders targeted": targeted,
                "ranking": name,
                "share of acquirers caught": caught[(targeted * len(order)).round().astype(int)],
            }
        )
    )
gains = pd.concat(curves)
fig = px.line(
    gains,
    x="share of non-holders targeted",
    y="share of acquirers caught",
    color="ranking",
    title="Card acquisition within 3 months: gains of each ranking (all months, 20% sample)",
)
fig.add_vline(x=0.1, line_dash="dot", line_color="grey").show()
top = gains[np.isclose(gains["share of non-holders targeted"], 0.1)].set_index("ranking")[
    "share of acquirers caught"
]
print(
    "Share of acquirers caught in the top 10%:", ", ".join(f"{k} {v:.0%}" for k, v in top.items())
)

Share of acquirers caught in the top 10%: oracle (true chance) 54%, best real column: income, highest first 25%, rule: salary transfer and income ≥ 5,000 16%, leak: preapproval score (peeks ahead) 89%, random 10%


In [17]:
cal = pd.DataFrame({"mean window probability": card_chance, "realised rate": y}).dropna()
cal = cal.groupby(
    pd.qcut(cal["mean window probability"].rank(method="first"), 10, labels=False)
).mean()
fig = px.scatter(
    cal,
    x="mean window probability",
    y="realised rate",
    title="Calibration, card acquisition within 3 months: realised rate per probability decile",
)
top_value = float(cal.to_numpy().max())
fig.add_shape(
    type="line", x0=0, y0=0, x1=top_value, y1=top_value, line_dash="dot", line_color="grey"
).show()

In [18]:
# Six card holders, low to high spenders: what they actually spent over the next 3 months, against
# the oracle's 10–90% band and median (simulated from each month's true state).
REG = "spend_cc_next_3_months"
fan = mart[["client_id", "snapshot_month"]].assign(realised=realised_target(mart, REG, 3))
fan = fan.merge(
    oracle[["client_id", "snapshot_month", f"{REG}_q10", f"{REG}_q50", f"{REG}_q90"]],
    on=["client_id", "snapshot_month"],
).dropna()
months_with_forecast = fan.groupby("client_id")["snapshot_month"].transform("size")
complete = fan[months_with_forecast == months_with_forecast.max()]
level = complete.groupby("client_id")[f"{REG}_q50"].mean().sort_values()
picked = level.index[(np.array([0.2, 0.4, 0.6, 0.75, 0.9, 0.98]) * (len(level) - 1)).astype(int)]
fig = make_subplots(
    rows=3, cols=2, subplot_titles=[f"client {c}" for c in picked], shared_xaxes=True
)
for i, client in enumerate(picked):
    one = complete[complete["client_id"] == client]
    row, col, legend = i // 2 + 1, i % 2 + 1, i == 0
    fig.add_trace(
        go.Scatter(x=one["snapshot_month"], y=one[f"{REG}_q90"], line_width=0, showlegend=False),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=one["snapshot_month"],
            y=one[f"{REG}_q10"],
            line_width=0,
            fill="tonexty",
            fillcolor="rgba(99,110,250,0.25)",
            name="oracle 10–90% band",
            showlegend=legend,
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=one["snapshot_month"],
            y=one[f"{REG}_q50"],
            line_color="rgb(99,110,250)",
            name="oracle median",
            showlegend=legend,
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=one["snapshot_month"],
            y=one["realised"],
            mode="markers",
            marker_color="black",
            name="realised",
            showlegend=legend,
        ),
        row=row,
        col=col,
    )
fig.update_layout(
    height=750, title="Next-3-month card spend (AED): the oracle's band against what happened"
)
fig.update_xaxes(title_text="observation month (spend over the next 3)", row=3)
fig.show()

## 8. Client stories
A few clients month by month: balances persist, card spend varies, and the hidden traits behind
them drift slowly.

In [19]:
some = mart["client_id"].drop_duplicates().sample(3, random_state=0)
path = mart[mart["client_id"].isin(some)]
px.line(
    path,
    x="snapshot_month",
    y="casa_bal",
    color="client_id",
    log_y=True,
    title="Monthly average current/savings balance (AED, log scale)",
).show()
px.line(
    path, x="snapshot_month", y="spend_cc_vol", color="client_id", title="Card spend a month (AED)"
).show()
traits = oracle[oracle["client_id"].isin(some)].melt(
    ["client_id", "snapshot_month"], [c for c in oracle.columns if c.startswith("z_")]
)
px.line(
    traits,
    x="snapshot_month",
    y="value",
    color="variable",
    facet_row="client_id",
    title="Hidden traits (oracle; standardised scale)",
).show()

---
## Appendix: checks and tables
The numbers behind the charts. **A1 first:** every check row must pass. The other parts back each
section with tables; every table says what its values are in the line above it or in its column
names.

### A1. Check tables
Every family's rows, written from the spec independently of the generator, as saved by `--checks`
(each family runs in its own process: at 100k one family needs several GB). `failed` must be 0
everywhere. The mart family reads every column, so it runs on the 20% client sample.

In [20]:
summary = pd.DataFrame(
    [
        (name, len(t), int(t["passed"].sum()), int((~t["passed"]).sum()), int(t["rule"].sum()))
        for name, t in tables.items()
    ],
    columns=["family", "rows", "passed", "failed", "rules (tolerance 0)"],
)
print("Check rows per family (counts)")
display(summary)
failed = pd.concat([t[~t["passed"]].assign(family=name) for name, t in tables.items()])
failed if len(failed) else print("Every row passes.")

Check rows per family (counts)


,family,rows,passed,failed,rules (tolerance 0)
0,population,158,158,0,7
1,holdings,440,440,0,52
2,amount_rules,112,112,0,112
3,amount_levels,45,45,0,0
4,seasons,27,27,0,0
5,mart,83,83,0,83
6,traps,103,103,0,15
7,oracle,341,341,0,138


Every row passes.


### A2. The book (section 1)

In [21]:
print("Share of clients per subsegment, first panel month")
display(mix.round(4).to_frame())
print("Share of each level within the profile, first panel month")
for column in ["nationality_group", "employment_status", "onboarding_channel"]:
    display(
        pd.crosstab(first["profile"], first[column], normalize="index").reindex(PROFILES).round(2)
    )
print("Age in years, quantiles per subsegment, first panel month")
ages = first.groupby("subsegment")["age"].quantile([0.1, 0.5, 0.9]).unstack().reindex(ORDER)
display(ages.set_axis(["age p10", "age p50", "age p90"], axis=1))
print("Declared monthly income (AED) per subsegment, first panel month")
income = first.groupby("subsegment")["income"].agg(
    **{
        "median declared income": "median",
        "share without declared income": lambda s: s.isna().mean(),
    }
)
display(income.reindex(ORDER).round(2))
print(
    "Mean monthly exit and entry rate per profile (share of the profile's clients, every client),"
)
print("and the exit rate as a share leaving per year")
mean = rates.groupby("profile")[["exit rate", "entry rate"]].mean().reindex(PROFILES)
display(mean.assign(**{"leaving per year": 1 - (1 - mean["exit rate"]) ** 12}).round(4))

Share of clients per subsegment, first panel month


,share of clients
subsegment,
Upper Private,0.0022
Lower Private,0.0075
Upper Priority,0.0094
Lower Priority,0.0288
UA,0.0448
A,0.0417
UM,0.2008
MM,0.6217
LM,0.0431


Share of each level within the profile, first panel month


nationality_group,AFRICA,ARAB,ASIA,GCC,OTHER,UAE,WESTERN
profile,,,,,,,
private,0.03,0.12,0.16,0.12,0.04,0.39,0.14
priority,0.02,0.10,0.31,0.08,0.02,0.17,0.30
affluent,0.05,0.13,0.50,0.03,0.03,0.15,0.10
mass,0.07,0.12,0.74,0.01,0.01,0.04,0.02


employment_status,Employed,Other,Retired,Self-employed,Unemployed
profile,,,,,
private,0.23,0.14,0.19,0.34,0.10
priority,0.55,0.08,0.06,0.23,0.08
affluent,0.76,0.04,0.03,0.13,0.04
mass,0.91,0.01,0.01,0.05,0.01


...

,age p10,age p50,age p90
subsegment,,,
Upper Private,39.0,52.5,66.0
Lower Private,38.0,53.0,71.7
Upper Priority,31.1,45.0,61.9
Lower Priority,30.0,42.0,57.0
UA,28.0,40.0,55.0
A,27.0,38.0,53.0
UM,26.0,36.0,49.0
MM,24.0,34.0,48.0
LM,23.0,32.0,45.0


Declared monthly income (AED) per subsegment, first panel month


,median declared income,share without declared income
subsegment,,
Upper Private,212060.0,0.67
Lower Private,87220.0,0.49
Upper Priority,58500.0,0.16
Lower Priority,42070.0,0.19
UA,32085.0,0.11
A,21230.0,0.09
UM,12160.0,0.05
MM,6040.0,0.06
LM,2950.0,0.04


Mean monthly exit and entry rate per profile (share of the profile's clients, every client),
and the exit rate as a share leaving per year


,exit rate,entry rate,leaving per year
profile,,,
private,0.0082,0.0084,0.0942
priority,0.0075,0.0107,0.0865
affluent,0.0090,0.0108,0.1030
mass,0.0111,0.0131,0.1251


### A3. Holdings (section 2)

In [22]:
ends = mart[mart["snapshot_month"].isin([months[0], months[-1]])]
ends = ends.assign(**{f"{k}_held": ends[c] > 0 for k, c in loans.items()})
held_columns = ["casa_flag", *flags, *[f"{k}_held" for k in loans]]
print(
    "Share of clients holding each product (flags: share with the flag at 1; loans: share with at"
)
print("least one account), per profile, first and last panel month")
display(ends.groupby(["profile", "snapshot_month"])[held_columns].mean().round(3))
print("Term-deposit status: share of client-months")
display(mart["td_status"].value_counts(normalize=True).round(3).to_frame("share of client-months"))
print(
    "Minimum declared income per product (AED):",
    {key: p["min_income"] for key, p in spec["products"].items() if "min_income" in p},
)

Share of clients holding each product (flags: share with the flag at 1; loans: share with at
least one account), per profile, first and last panel month


casa_flag  casa_active_flag  cc_flag  cc_active_flag  td_flag  wealth_digital_flag  wealth_trad_flag  ebanking_flag  digital_active_flag  fx_active_flag  loan_personal_held  \
profile  snapshot_month                                                                                                                                                                                 
affluent 2024-10-01          0.992             0.932    0.652           0.509    0.150                0.114             0.085          0.882                0.708           0.294               0.205   
         2026-09-01          0.990             0.930    0.662           0.520    0.144                0.116             0.089          0.894                0.744           0.315               0.206   
mass     2024-10-01          0.989             0.951    0.299           0.217    0.041                0.020             0.010          0.746                0.584           0.118               0.202   
         2026-09-01          0.989             0.952    0.304           0.218    0.041                0.021             0.009          0.750                0.613           0.119               0.203   
priority 2024-10-01          0.989             0.925    0.783           0.642    0.312                0.185             0.329          0.899                0.714           0.500               0.117   
         2026-09-01          0.988             0.927    0.811           0.668    0.313                0.202             0.320          0.885                0.736           0.484               0.115   
private  2024-10-01          0.995             0.885    0.845           0.740    0.445                0.095             0.575          0.825                0.630           0.590               0.045   
         2026-09-01          0.995             0.915    0.839           0.688    0.432                0.126             0.573          0.794                0.658           0.678               0.045   

                         loan_auto_held  loan_home_held  
profile  snapshot_month                                  
affluent 2024-10-01               0.122           0.111  
         2026-09-01               0.127           0.107  
mass     2024-10-01               0.058           0.020  
         2026-09-01               0.062           0.018  
priority 2024-10-01               0.089           0.223  
         2026-09-01               0.093           0.217  
private  2024-10-01               0.035           0.175  
         2026-09-01               0.035           0.191

Term-deposit status: share of client-months


,share of client-months
td_status,
never,0.843
churned,0.093
active,0.064


Minimum declared income per product (AED): {'cc': 5000, 'loan_personal': 5000, 'loan_auto': 5000, 'loan_home': 15000}


### A4. Amounts (section 3)
The booking identity must hold exactly: inflow − outflow equals the change in the balance.

In [23]:
amount_cols = [
    "casa_bal",
    "inflow_casa_vol",
    "outflow_casa_vol",
    "spend_cc_vol",
    "cc_limit",
    "cc_utilisation_ratio",
    "td_bal",
    "wealth_trad_bal",
    "wealth_digital_bal",
    "loan_personal_bal",
    "loan_home_bal",
    "fx_vol",
    "login_digital_mobile_cnt",
]
print("Median of each amount among clients with a value above 0, May 2025, per subsegment")
print("(AED; utilisation a ratio; logins a count)")
medians = may.groupby("subsegment")[amount_cols].agg(lambda s: s[s > 0].median()).reindex(ORDER)
display(medians.round(2).rename(columns=lambda c: f"median {c}"))
print("Card limit tiers (AED):", spec["levels"]["card"]["limit_tiers"])
print("Share of cards per limit tier, per profile, May 2025 (every card)")
display(
    pd.crosstab(cards["profile"], cards["cc_limit"], normalize="index").reindex(PROFILES).round(3)
)
ordered = mart[
    [
        "client_id",
        "snapshot_month",
        "casa_bal",
        "inflow_casa_vol",
        "outflow_casa_vol",
        "casa_active_flag",
    ]
].sort_values(["client_id", "snapshot_month"])
# Rows with a previous row in the panel (a starting-book client's first panel row continues the
# burn-in).
change = ordered["casa_bal"] - ordered.groupby("client_id")["casa_bal"].shift()
gap = (ordered["inflow_casa_vol"] - ordered["outflow_casa_vol"] - change)[
    ordered["casa_active_flag"] == 1
]
print(f"booking identity, largest |inflow − outflow − balance change| (AED): {gap.abs().max():.4f}")
for total, columns in parts.items():
    print(f"Share of each subcategory in the profile's total {total} (all panel months)")
    sums = mart.groupby("profile")[columns].sum()
    display(sums.div(sums.sum(axis=1), axis=0).reindex(PROFILES).round(3))
print("Remittance per nationality group: share of client-months with a remittance, and the mean")
print("remittance per client-month (AED, zeros included)")
display(
    mart.groupby("nationality_group")["outflow_casa_remittance_vol"]
    .agg(**{"share remitting": lambda s: (s > 0).mean(), "mean remittance": "mean"})
    .round(2)
)
um_median = mm_um.loc[mm_um["subsegment"] == "UM", "casa_bal"].median()
above = (mm_um.loc[mm_um["subsegment"] == "MM", "casa_bal"] > um_median).mean()
print(f"share of MM clients above the UM median balance, May 2025: {above:.3f}")

Median of each amount among clients with a value above 0, May 2025, per subsegment
(AED; utilisation a ratio; logins a count)


,median casa_bal,median inflow_casa_vol,median outflow_casa_vol,median spend_cc_vol,median cc_limit,median cc_utilisation_ratio,median td_bal,median wealth_trad_bal,median wealth_digital_bal,median loan_personal_bal,median loan_home_bal,median fx_vol,median login_digital_mobile_cnt
subsegment,,,,,,,,,,,,,
Upper Private,4339692.79,268474.53,346385.76,67819.67,250000.0,0.22,4187721.77,12882591.85,2731345.79,199077.06,9096683.64,210471.74,6.0
Lower Private,2085066.10,166409.77,111446.48,26158.54,150000.0,0.20,1308706.09,7872837.78,730047.45,265667.25,4152997.46,82678.92,5.0
Upper Priority,153217.79,69538.18,64499.79,13675.45,100000.0,0.19,538751.79,1104950.54,301896.40,449858.70,4061771.70,37610.14,10.0
Lower Priority,100096.22,45613.92,45620.72,9949.06,75000.0,0.19,442243.51,649312.51,159559.30,185009.42,2833127.65,22359.64,9.0
UA,34218.55,37372.76,36903.17,8237.54,50000.0,0.24,210944.87,252450.00,72872.78,151878.70,2013466.24,12487.93,14.0
A,23702.64,23793.00,23925.90,4750.22,35000.0,0.27,124303.87,172722.50,66100.53,107069.92,1322452.68,9093.28,14.0
UM,4987.12,13507.67,13333.22,2851.97,10000.0,0.48,35033.94,54094.66,15124.94,67258.63,883555.40,3731.14,20.0
MM,2447.11,6343.88,6342.78,1579.23,10000.0,0.49,20712.45,31324.40,7135.93,40229.72,354875.87,2011.24,20.0
LM,1225.92,2938.90,2997.39,774.12,5000.0,0.38,10000.00,14363.19,2016.43,18069.54,145873.00,1205.93,21.0


Card limit tiers (AED): [5000, 10000, 20000, 35000, 50000, 75000, 100000, 150000, 250000, 500000]
Share of cards per limit tier, per profile, May 2025 (every card)


cc_limit,5000.0,10000.0,20000.0,35000.0,50000.0,75000.0,100000.0,150000.0,250000.0,500000.0
profile,,,,,,,,,,
private,0.000,0.006,0.024,0.030,0.065,0.030,0.065,0.232,0.345,0.202
priority,0.011,0.045,0.030,0.046,0.137,0.185,0.278,0.185,0.080,0.002
affluent,0.011,0.044,0.165,0.243,0.286,0.152,0.077,0.021,0.002,0.000
mass,0.351,0.459,0.157,0.026,0.005,0.001,0.000,0.000,0.000,0.000


booking identity, largest |inflow − outflow − balance change| (AED): 0.0000
Share of each subcategory in the profile's total outflow (all panel months)


...

Share of each subcategory in the profile's total card spend (all panel months)


,spend_cc_travel_vol,spend_cc_retail_vol,spend_cc_online_vol,spend_cc_dining_vol
profile,,,,
private,0.314,0.307,0.194,0.185
priority,0.179,0.368,0.246,0.207
affluent,0.084,0.439,0.281,0.197
mass,0.009,0.514,0.270,0.206


Remittance per nationality group: share of client-months with a remittance, and the mean
remittance per client-month (AED, zeros included)


,share remitting,mean remittance
nationality_group,,
AFRICA,0.55,1456.21
ARAB,0.55,1397.13
ASIA,0.56,1524.92
GCC,0.00,0.00
OTHER,0.51,1397.41
UAE,0.00,0.00
WESTERN,0.46,1013.30


share of MM clients above the UM median balance, May 2025: 0.244


### A5. The mart (section 5)
A null shows as `<NA>` in integer columns and `NaN` in float columns. Both are true nulls: pandas'
nullable integers display missing values as `<NA>`, the parquet files store them as nulls, and Spark
reads them as `null` (`tests/test_synthetic_mart.py`).

In [24]:
print("Mart columns per kind (counts)")
display(kinds.value_counts().to_frame("columns"))
print("Share of rows that are null, per column with any null (20% sample, all months)")
display(nulls.sort_values("share null", ascending=False).round(3))
# One client with a card who spends in some months only, so its recency column takes values.
per_client = mart.groupby("client_id").agg(
    months=("snapshot_month", "size"), spending=("spend_cc_vol", lambda s: (s > 0).mean())
)
example_id = per_client[
    (per_client["months"] == 24) & per_client["spending"].between(0.4, 0.8)
].index[0]
example = mart[mart["client_id"] == example_id]
print(f"Client {example_id}: monthly average balance (AED), its 3- and 6-month rollups, card spend")
print("(AED) and months since the last month with card spend (0 = spent this month)")
display(
    example[
        [
            "snapshot_month",
            "casa_bal",
            "casa_bal_0_2_months_avg",
            "casa_bal_0_5_months_avg",
            "casa_bal_0_5_months_std",
            "spend_cc_vol",
            "months_since_last_spend_cc",
            "relationship_days",
        ]
    ]
)
column = "months_since_last_spend_cc"
values = mart[column]
print(
    f"{column}: pandas dtype {values.dtype}; {values.isna().sum():,} nulls; "
    f"{values.map(lambda v: isinstance(v, str)).sum()} string values"
)
part = pq.read_table(
    next((DATA / "mart").glob("snapshot_month=*/part-0.parquet")), columns=[column]
)
stored, n_null = part.schema.field(column).type, part.column(column).null_count
print(f"in the parquet file: type {stored}, {n_null:,} nulls of {part.num_rows:,} rows (one month)")

Mart columns per kind (counts)


,columns
kind,
rollup,65
holding,20
flow,20
trap,20
stock,10
count,8
categorical,7
static,5
id,1


Share of rows that are null, per column with any null (20% sample, all months)


,share null,kind
legacy_risk_grade_trap_all_null,1.000,trap
complaint_category_trap_low_fill,0.997,trap
loyalty_points_bal_trap_low_fill,0.990,trap
cc_offer_code_trap_missing_leak,0.980,trap
loan_home_maturity_months,0.964,holding
loan_auto_maturity_months,0.933,holding
loan_personal_maturity_months,0.802,holding
months_since_last_fx,0.748,rollup
spend_cc_vol_0_2_vs_3_5_months_ratio,0.719,rollup
cc_utilisation_ratio,0.643,stock


Client 5: monthly average balance (AED), its 3- and 6-month rollups, card spend
(AED) and months since the last month with card spend (0 = spent this month)


,snapshot_month,casa_bal,casa_bal_0_2_months_avg,casa_bal_0_5_months_avg,casa_bal_0_5_months_std,spend_cc_vol,months_since_last_spend_cc,relationship_days
0,2024-10-01,5853.33,5333.283333,5507.568333,499.695106,1609.61,0,754
20528,2024-11-01,5799.24,5703.560000,5451.615000,429.218686,2070.27,0,784
41114,2024-12-01,7310.61,6321.060000,5732.851667,880.114022,2665.44,0,815
61739,2025-01-01,5839.87,6316.573333,5824.928333,852.662820,1071.77,0,846
82394,2025-02-01,6071.49,6407.323333,6055.441667,645.828116,4383.82,0,874
103111,2025-03-01,6575.87,6162.410000,6241.735000,598.544747,0.00,1,905
123876,2025-04-01,5350.17,5999.176667,6157.875000,691.825669,0.00,2,935
144692,2025-05-01,4590.05,5505.363333,5956.343333,946.454423,0.00,3,966
165560,2025-06-01,5364.53,5101.583333,5631.996667,687.587104,0.00,4,996
186469,2025-07-01,5516.41,5156.996667,5578.086667,680.674864,0.00,5,1027


months_since_last_spend_cc: pandas dtype Int64; 304,192 nulls; 0 string values
in the parquet file: type int64, 63,692 nulls of 104,252 rows (one month)


### A6. Traps (section 6)
The trap summary, then the evidence. `trap_column` True: a column added only to test the
framework, its name ending in `_trap_<kind>`. False: the trap sits in real columns (named in
`column`; none for collinearity and dirty data), which keep their real names because the framework
meets the same columns in the real mart.

In [25]:
print("The trap summary (truth.json: spec.traps)")
display(trap_summary(spec)[["product", "kind", "trap_column", "column", "expected"]])
trap_rows = tables["traps"]
print("Evidence: the trap family's rows for the leaks, the upstream target, the process leaks and")
print("shifted columns, the rare level, Simpson's reversal, the null controls, the constants, the")
print("redundant column, the low-fill columns and the drift")
display(
    trap_rows[
        trap_rows["check"].str.contains(
            "leak|upstream|shifted|Ghana|Simpson|null control|constant|redundant|low fill|PSI",
            case=False,
        )
    ][["check", "stratum", "expected", "realised", "tolerance", "passed"]]
)
print("Leakage screen behind section 6: the oracle's AUC and the 10 strongest columns per target")
display(pd.concat({t: s.nlargest(10, "strength") for t, s in screens.items()}).round(3))

The trap summary (truth.json: spec.traps)


,product,kind,trap_column,column,expected
0,cc,leak,True,cc_preapproval_score_trap_leak,univariate AUC ≈ 0.97 for card acquisition in ...
1,cc,leak,True,cc_campaign_response_score_trap_leak,univariate AUC ≈ 0.875 for card acquisition in...
2,cc,upstream_target,True,cc_acquire_next_3_months_flag_trap_upstream_ta...,equals Target Factory's own target except leav...
3,cc,simpson,False,login_digital_mobile_cnt,logins vs card acquisition: positive within ea...
4,cc,null_income_eligibility,False,income,a plain income filter silently drops exempt cl...
5,cc,onboarding,False,relationship_days,most adoption in the first 3 months: the stron...
6,cc,eligibility_defining,False,spend_cc_vol,"constant (0) within non-holders, the eligible ..."
7,cc,decimal_type,False,cc_limit,Spark reads a decimal; pandas gets Python obje...
8,cc,missing_leak,True,cc_offer_code_trap_missing_leak,filled for 80% of non-holders who take the car...
9,cc,process_leak,True,application_cc_cnt_trap_process_leak,beats the oracle for acquisitions in the next ...


Evidence: the trap family's rows for the leaks, the upstream target, the process leaks and
shifted columns, the rare level, Simpson's reversal, the null controls, the constants, the
redundant column, the low-fill columns and the drift


,check,stratum,expected,realised,tolerance,passed
0,leak cc_preapproval_score_trap_leak: AUC for c...,"w=3, one month",0.970000,0.966914,0.012336,True
1,leak cc_preapproval_score_trap_leak: AUC above...,"w=3, one month",0.841854,0.966914,NaN,True
2,leak cc_campaign_response_score_trap_leak: AUC...,"w=3, one month",0.875000,0.876865,0.023346,True
3,leak cc_campaign_response_score_trap_leak: AUC...,"w=3, one month",0.841854,0.876865,NaN,True
4,leaks: correlation among non-acquirers (indepe...,"w=3, one month",0.000000,0.007170,0.017838,True
5,missing leak: cc_offer_code_trap_missing_leak ...,all,0.800000,0.800766,0.009008,True
6,missing leak: cc_offer_code_trap_missing_leak ...,all,0.010000,0.010096,0.000388,True
7,missing leak: cc_offer_code_trap_missing_leak ...,all,0.000000,0.000000,0.000000,True
8,missing leak: cc_offer_code_trap_missing_leak ...,all,0.000000,0.000000,0.000000,True
9,missing leak: AUC of 'cc_offer_code_trap_missi...,"w=3, one month",0.841854,0.893728,NaN,True


Leakage screen behind section 6: the oracle's AUC and the 10 strongest columns per target


column    AUC  strength         kind  oracle
cc_acquire         115  cc_acquire_next_3_months_flag_trap_upstream_ta...  1.000     1.000  trap column   0.842
                   113                     cc_preapproval_score_trap_leak  0.970     0.970  trap column   0.842
                   126          cc_offer_code_trap_missing_leak is filled  0.896     0.896  trap column   0.842
                   114               cc_campaign_response_score_trap_leak  0.873     0.873  trap column   0.842
                   119                              cc_limit_trap_shifted  0.704     0.704  trap column   0.842
                   0                                               income  0.679     0.679  real column   0.842
                   59                     outflow_casa_vol_0_5_months_avg  0.676     0.676  real column   0.842
                   53                      inflow_casa_vol_0_5_months_avg  0.676     0.676  real column   0.842
                   58                     outflow_casa_vol_0_2_months_avg  0.675     0.675  real column   0.842
                   52                      inflow_casa_vol_0_2_months_avg  0.674     0.674  real column   0.842
loan_personal_open 129                     loan_personal_bal_trap_shifted  0.944     0.944  trap column   0.763
                   128    application_loan_personal_cnt_trap_process_leak  0.900     0.900  trap column   0.763
                   67               inflow_casa_salary_vol_0_5_months_avg  0.642     0.642  real column   0.763
                   66               inflow_casa_salary_vol_0_2_months_avg  0.640     0.640  real column   0.763
                   24                              inflow_casa_salary_vol  0.638     0.638  real column   0.763
                   75                outflow_casa_cash_vol_0_5_months_avg  0.606     0.606  real column   0.763
                   74                outflow_casa_cash_vol_0_2_months_avg  0.603     0.603  real column   0.763
                   65                      inflow_casa_vol_0_5_months_avg  0.603     0.603  real column   0.763
                   71                     outflow_casa_vol_0_5_months_avg  0.603     0.603  real column   0.763
                   64                      inflow_casa_vol_0_2_months_avg  0.603     0.603  real column   0.763

### A7. The ceiling (section 7)
The oracle's AUC per target (windows of 1, 3 and 6 months) and its sizing; the regression oracle's
coverage; then single-column AUCs for the activation targets: how far each mart column alone gets,
against the oracle.

In [26]:
oracle_auc = pd.DataFrame(
    {
        t: {
            "oracle AUC w=1": m["w1"]["auc"],
            "oracle AUC w=3": m["w3"]["auc"],
            "oracle AUC w=6": m["w6"]["auc"],
            "positive rows a month (w=3)": m["w3"]["positive_rows_per_month"],
            "expected events in 6 months": m["expected_events_6m"],
            "low confidence": m["low_confidence"],
        }
        for t, m in truth["oracle"].items()
    }
).T
print("Per standard target: the oracle's AUC (all rows, all months, every client), positive rows a")
print("month, sizing")
display(oracle_auc.round(3))
oracle_rows = tables["oracle"]
print(
    "Regression oracle (next-3-month card spend): coverage of its quantiles and the ratio of means"
)
display(
    oracle_rows[oracle_rows["check"].str.contains(REG, case=False)][
        ["check", "stratum", "expected", "realised", "tolerance", "passed"]
    ]
)

Per standard target: the oracle's AUC (all rows, all months, every client), positive rows a
month, sizing


,oracle AUC w=1,oracle AUC w=3,oracle AUC w=6,positive rows a month (w=3),expected events in 6 months,low confidence
cc_acquire,0.866691,0.845755,0.833094,1901.333333,4716.960857,False
td_acquire,0.747465,0.744742,0.742579,826.333333,1664.684716,False
wealth_digital_acquire,0.794901,0.793363,0.790511,1245.904762,2579.641739,False
wealth_trad_acquire,0.885013,0.882011,0.880958,134.142857,276.745602,False
ebanking_acquire,0.950727,0.925684,0.899609,904.857143,3211.555618,False
loan_personal_open,0.765953,0.765965,0.767836,2105.47619,4360.178615,False
loan_auto_open,0.717806,0.71551,0.714081,505.857143,1023.625829,False
loan_home_open,0.829866,0.828261,0.827623,139.714286,291.628521,False
fx_activate,0.744419,0.742687,0.739642,1736.571429,3540.436233,False
digital_activate,0.826436,0.794977,0.799004,5140.333333,15930.131098,False


Regression oracle (next-3-month card spend): coverage of its quantiles and the ratio of means


,check,stratum,expected,realised,tolerance,passed
294,spend_cc_next_3_months: P(realised < q10),observation 2024-11,0.111314,0.078237,NaN,True
295,"spend_cc_next_3_months: P(realised < q10), q10...",observation 2024-11,0.103980,0.104366,0.008471,True
296,"spend_cc_next_3_months: P(realised ≤ q50), q50...",observation 2024-11,0.500000,0.501349,0.013493,True
297,spend_cc_next_3_months: P(realised ≤ q90),observation 2024-11,0.888686,0.896273,NaN,True
298,spend_cc_next_3_months: realised / predicted mean,observation 2024-11,1.000000,0.998102,0.024531,True
299,spend_cc_next_3_months: P(realised < q10),observation 2024-12,0.111307,0.075973,NaN,True
300,"spend_cc_next_3_months: P(realised < q10), q10...",observation 2024-12,0.103980,0.101271,0.008459,True
301,"spend_cc_next_3_months: P(realised ≤ q50), q50...",observation 2024-12,0.500000,0.502010,0.013479,True
302,spend_cc_next_3_months: P(realised ≤ q90),observation 2024-12,0.888693,0.898902,NaN,True
303,spend_cc_next_3_months: realised / predicted mean,observation 2024-12,1.000000,0.994191,0.023659,True


In [27]:
# Single-column AUCs: every numeric mart column alone against the realised target (w = 3), in the
# middle panel month (one row per client), next to the oracle's window probability on the same rows.
# "strength" is max(AUC, 1 − AUC); columns with a value in fewer than 300 eligible rows are left
# out.
at_obs = (mart["snapshot_month"] == months[11]).to_numpy()
rows = []
for target in ["cc_activate", "digital_activate", "casa_reactivate", "fx_activate"]:
    y = realised_target(mart, target, 3)
    use = at_obs & y.notna().to_numpy()
    labels = y[use].to_numpy()
    probability = window_probability(mart, oracle, truth, target, 3).to_numpy()[use]
    rows.append(
        {
            "target": target,
            "column": "ORACLE (window probability)",
            "AUC": auc(probability, labels),
            "rows": int(use.sum()),
            "events": int(labels.sum()),
        }
    )
    for column in numeric:
        values = mart.loc[use, column].to_numpy(dtype=float, na_value=np.nan)
        known = ~np.isnan(values)
        if known.sum() >= 300 and np.unique(values[known]).size > 1:  # sparse columns: noise
            rows.append(
                {
                    "target": target,
                    "column": column,
                    "AUC": auc(values[known], labels[known]),
                    "rows": int(known.sum()),
                    "events": int(labels[known].sum()),
                }
            )
single = pd.DataFrame(rows).assign(strength=lambda d: np.maximum(d["AUC"], 1 - d["AUC"]))
print(f"Top 8 single columns per activation target by strength, {months[11]:%Y-%m}, 20% sample")
display(
    single.sort_values(["target", "strength"], ascending=[True, False])
    .groupby("target")
    .head(9)
    .round(3)
    .set_index(["target", "column"])
)

Top 8 single columns per activation target by strength, 2025-09, 20% sample


AUC   rows  events  strength
target           column                                                                 
casa_reactivate  ORACLE (window probability)              0.679    864     305     0.679
                 inflow_casa_salary_vol_0_5_months_avg    0.601    809     280     0.601
                 salary_transfer_flag                     0.600    864     305     0.600
                 outflow_casa_cash_vol_0_5_months_avg     0.594    809     280     0.594
                 inflow_casa_vol_0_5_months_avg           0.591    809     280     0.591
                 outflow_casa_vol_0_5_months_avg          0.590    809     280     0.590
                 inflow_casa_vol_0_5_months_max           0.590    809     280     0.590
                 inflow_casa_vol_0_5_months_std           0.589    809     280     0.589
                 outflow_casa_bills_vol_0_5_months_avg    0.587    809     280     0.587
cc_activate      ORACLE (window probability)              0.702   1761     682     0.702
                 inflow_casa_transfers_vol                0.580   1761     682     0.580
                 inflow_casa_vol_0_5_months_std           0.572   1616     605     0.572
                 inflow_casa_other_vol_0_2_months_avg     0.571   1685     635     0.571
                 inflow_casa_other_vol                    0.571   1761     682     0.571
                 outflow_casa_cnt_0_5_months_avg          0.571   1616     605     0.571
                 inflow_casa_other_vol_0_5_months_avg     0.569   1616     605     0.569
                 inflow_casa_vol_0_5_months_max           0.557   1616     605     0.557
                 cc_limit_trap_shifted                    0.557   1761     682     0.557
digital_activate ORACLE (window probability)              0.801   3220    1064     0.801
                 days_since_last_login_digital            0.389   2653     796     0.611
                 login_digital_mobile_cnt_0_5_months_std  0.609   2905     857     0.609
                 login_digital_mobile_cnt_0_5_months_max  0.608   2905     857     0.608
                 login_digital_mobile_cnt_0_5_months_avg  0.606   2905     857     0.606
                 login_digital_online_cnt_0_5_months_avg  0.595   2905     857     0.595
                 relationship_days                        0.409   3220    1064     0.591
                 relationship_months_trap_redundant       0.409   3220    1064     0.591
                 login_digital_mobile_cnt_0_5_months_nz   0.589   2905     857     0.589
fx_activate      ORACLE (window probability)              0.737  17268     330     0.737
                 income                                   0.626  16197     297     0.626
                 casa_bal_0_5_months_min                  0.617  16194     311     0.617
                 casa_bal                                 0.616  17268     330     0.616
                 casa_bal_0_5_months_avg                  0.616  16194     311     0.616
                 casa_bal_0_5_months_max                  0.616  16194     311     0.616
                 casa_bal_0_2_months_avg                  0.615  16823     322     0.615
                 inflow_casa_vol_0_5_months_avg           0.601  16194     311     0.601
                 outflow_casa_vol_0_5_months_avg          0.601  16194     311     0.601